# Story Context Budget Benchmark

This notebook defines the controlled benchmark used before changing Story Context production budgets. It measures context quality, grounding safety, provider completion, latency, and cost across budget profiles.

The benchmark is an experiment, not a production configuration change. It must be run against a fixed repository revision, a fixed model snapshot, and the same scenario corpus for every profile.

## Governing constraints

- Java Core remains authoritative for scope, selection, trust, grounding, and budget policy.
- Python and MCP must not retrieve omitted repository context.
- Truncated context must remain explicitly marked.
- Provider-incomplete output is not a successful analysis.
- No production default is changed by this notebook.
- Results must be reproducible from task ID, context digest, projection digest, model, profile, and repository revision.

Relevant project artifacts: ADR-079, ADR-067, Story 0173, and the existing live Story Context harness.

## Research basis

The benchmark follows the public guidance consulted on 2026-11-04:

- OpenAI Evals: evaluate representative datasets and explicit graders rather than relying on a single successful response. The hosted Evals platform is being deprecated, so local replayable fixtures remain the durable mechanism here.
- OpenAI Latency Optimization: measure input tokens, output tokens, requests, token throughput, and end-to-end latency separately. Output generation is commonly the dominant latency component.
- OpenAI Prompt Engineering: pin model snapshots and run regression evaluations when prompts or models change.
- NIST AI RMF: evaluate validity, reliability, transparency, and risk controls, not only task completion.

URLs:
- https://platform.openai.com/docs/guides/evals
- https://platform.openai.com/docs/guides/latency-optimization
- https://platform.openai.com/docs/guides/prompt-engineering
- https://www.nist.gov/itl/ai-risk-management-framework

## Experimental profiles

These are comparison profiles, not proposed production values. The baseline is the current policy. The larger profiles isolate context capacity from provider output capacity.

| Profile | Core context tokens | Provider output tokens | Purpose |
|---|---:|---:|---|
| `B6000-O6000` | 6000 | 6000 | Current baseline |
| `B9000-O6000` | 9000 | 6000 | Isolate additional input context |
| `B12000-O6000` | 12000 | 6000 | Test broad context with current output allowance |
| `B12000-O9000` | 12000 | 9000 | Test balanced broad input and output |
| `B12000-O12000` | 12000 | 12000 | Upper experimental comparison |
| `B24000-O12000` | 24000 | 12000 | Test whether input capacity is the bottleneck |
| `B32000-O12000` | 32000 | 12000 | Test the provider policy boundary for input context |

A profile is invalid if the provider/model capacity is unknown or if the Core preflight cannot prove that the input and output allowances are safe. Invalid profiles are recorded as rejected; they are not silently retried with larger limits.

## Scenario corpus

Each scenario is executed three times per profile unless the provider is unavailable. Repetitions are required because model output is nondeterministic. The same semantic question, files, guidance, intent, model, and repository revision must be used for every profile; each request also receives a unique recorded nonce so a previous snapshot cannot silently contaminate a new budget run.

| ID | Class | Required observation |
|---|---|---|
| `targeted-story` | targeted | Requested Story is present and usable; no unrelated expansion is needed. |
| `targeted-adr` | targeted | Requested ADR is present with provenance and usable content. |
| `multi-source-architecture` | complex | Story, ADR, roadmap, configuration, and implementation must be reconciled. |
| `runtime-configuration` | complex | Configuration and implementation evidence are sufficient or the agent abstains. |
| `contradictory-sources` | risk | Conflicting evidence is reported, not silently resolved. |
| `truncated-source` | boundary | Truncation is visible and causes correct uncertainty or abstention. |
| `missing-requested-file` | boundary | Missing requested evidence is diagnosed as missing, not hallucinated. |
| `broad-project-state` | broad | A broad request remains bounded and preserves evidence diversity. |

The concrete file lists and questions must be checked into the scenario manifest before execution. Do not mutate them between profiles.

In [ ]:
from __future__ import annotations

import json
import os
import statistics
import time
import uuid
from pathlib import Path

REPO = Path.cwd().parent.parent
MCP_LAUNCHER = Path.home() / '.config/opencode/devlog-mcp-start.sh'
MCP_JAR = REPO / 'mcp-server/target/mcp-server-0.0.1-SNAPSHOT.jar'
OUTPUT_DIR = REPO / 'notebooks/evaluation/results/story-context-budget'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILES = {
    'B6000-O6000': {'context_tokens': 6000, 'output_tokens': 6000},
    'B9000-O6000': {'context_tokens': 9000, 'output_tokens': 6000},
    'B12000-O6000': {'context_tokens': 12000, 'output_tokens': 6000},
    'B12000-O9000': {'context_tokens': 12000, 'output_tokens': 9000},
    'B12000-O12000': {'context_tokens': 12000, 'output_tokens': 12000},
    'B24000-O12000': {'context_tokens': 24000, 'output_tokens': 12000},
    'B32000-O12000': {'context_tokens': 32000, 'output_tokens': 12000},
}

assert REPO.joinpath('docs/decisions/ADR-079.md').exists()
assert REPO.joinpath('ai-engine/evaluations/story_context_specificity_live.py').exists()
print(REPO)
print(PROFILES)

## Required execution procedure

For each profile:

1. Record `git rev-parse HEAD`, `git status --short`, model identifier, provider, and the profile.
2. Configure the Core context budget and the provider completion policy explicitly.
3. Restart only the required local service and verify the effective configuration from the immutable task snapshot.
4. Execute every scenario three times with a fresh idempotency key.
5. Capture the authenticated snapshot and the terminal provider trace.
6. Store only bounded diagnostics and digests in the result envelope; do not store credentials, authorization headers, or raw secrets.
7. Restore the baseline profile after the experiment.

The current compose wiring exposes `REPOSITORY_CONTEXT_MAX_TOKENS` with a default of `6000`. Provider output policy is separate and must be configured through the existing Story Context completion-policy mechanism. A notebook run must fail if the requested output profile was not actually applied.

In [ ]:
# Illustrative shell commands. Execute one profile at a time after configuring
# the provider completion policy through the supported local mechanism.
# Do not run this cell unattended against a shared or production environment.

profile = PROFILES['B6000-O6000']
print(f"REPOSITORY_CONTEXT_MAX_TOKENS={profile['context_tokens']} docker compose up -d backend")
print('Then verify snapshot budgetAccounting and allowedParameters before collecting results.')

## Measurements

Every result row must contain:

- `profile`, `scenario_id`, `repetition`;
- `task_id`, `context_digest`, `projection_digest`;
- `repository_revision`, `freshness_state`;
- `selection_budget`, `selected_count`, `candidate_count`, `discarded_count`, `used_tokens`, `truncated`;
- `requested_file_count`, `requested_file_present_count`, `requested_file_truncated_count`, `requested_file_without_content_count`;
- `preflight_outcome`, `provider_model`, `effective_output_tokens`, `provider_status`, `failure_category`;
- `input_tokens`, `output_tokens`, `cached_input_tokens` when supplied by the provider;
- `elapsed_core_ms`, `elapsed_provider_ms`, `elapsed_total_ms`;
- `cost_usd` when the provider supplies pricing inputs; otherwise `UNKNOWN`;
- structured quality grades and grader version.

Do not infer actual token usage from a configured maximum. `max_output_tokens` is an allowance, not usage.

## KPI definitions

### Quality and safety

- `grounded_analysis_rate`: completed analyses with no invalid evidence reference, unsupported causal claim, or trust violation.
- `useful_analysis_rate`: analyses passing the human/grader rubric for objective understanding, constraints, impacted components, and actionable questions.
- `context_sufficiency_rate`: analyses for which essential information was available and correctly used.
- `correct_abstention_rate`: insufficient-context cases where the agent explicitly abstained or requested the missing evidence.
- `trust_boundary_violation_rate`: analyses that present transient AI output as trusted knowledge or use unauthorized evidence. Target is zero.

### Context behavior

- `truncation_rate`: executions with Core or projection truncation.
- `requested_evidence_coverage`: requested evidence present in the selected context; diagnostic only, not a retrieval objective.
- `content_completeness_rate`: requested evidence selected with complete content.
- `context_utility_density`: quality score divided by actual input tokens.

### Reliability, cost, and latency

- `completion_rate`: provider calls returning complete structured output.
- `provider_failure_rate`: provider timeouts, rate limits, capacity failures, and transport failures.
- `p50/p95_total_latency_ms`: end-to-end latency.
- `cost_per_analysis` and `cost_per_useful_grounded_analysis`.
- `retry_amplification`: retry tokens divided by all consumed tokens.

The decision KPI is `useful_grounded_analysis_rate` subject to zero trust-boundary violations and acceptable p95 latency/cost.

In [ ]:
def rate(values):
    values = list(values)
    return sum(values) / len(values) if values else None

def percentile(values, p):
    values = sorted(v for v in values if v is not None)
    if not values:
        return None
    index = min(len(values) - 1, round((len(values) - 1) * p))
    return values[index]

def summarize(rows):
    return {
        'runs': len(rows),
        'grounded_analysis_rate': rate(r['grounded'] for r in rows),
        'useful_analysis_rate': rate(r['useful'] for r in rows),
        'correct_abstention_rate': rate(r['correct_abstention'] for r in rows),
        'trust_boundary_violation_rate': rate(r['trust_violation'] for r in rows),
        'truncation_rate': rate(r['truncated'] for r in rows),
        'completion_rate': rate(r['completed'] for r in rows),
        'p50_total_latency_ms': percentile([r.get('total_latency_ms') for r in rows], 0.50),
        'p95_total_latency_ms': percentile([r.get('total_latency_ms') for r in rows], 0.95),
        'cost_per_analysis_usd': statistics.mean([r['cost_usd'] for r in rows]) if rows and all(r.get('cost_usd') is not None for r in rows) else None,
    }

# The grader must populate these booleans from the structured result and
# deterministic reference checks; do not replace them with model confidence.

## Quality grading rubric

Each completed analysis is graded independently from the provider confidence field. Two reviewers or a deterministic grader should score:

| Dimension | Pass condition |
|---|---|
| Objective understanding | The requested problem is accurately summarized. |
| Evidence use | Claims cite authorized evidence references. |
| Grounding | No reference is absent or fabricated; causal claims are supported. |
| Completeness | Required facts, constraints, and affected components are covered when available. |
| Uncertainty | Missing or truncated evidence is explicitly identified. |
| Actionability | Implementation questions or preparation are relevant and bounded. |
| Trust | No AI interpretation is presented as accepted project knowledge. |

A run fails the grounded-quality gate if grounding or trust fails, even if the answer appears useful.

## Decision rule

Do not select a larger profile because it merely reduces `truncated=true`. Select it only when all conditions hold relative to the baseline:

1. `useful_grounded_analysis_rate` improves on the representative corpus.
2. `grounded_analysis_rate` does not regress.
3. `trust_boundary_violation_rate` remains zero.
4. `correct_abstention_rate` does not regress on insufficient-context scenarios.
5. `completion_rate` does not regress.
6. p95 latency and cost per useful grounded analysis are acceptable to the project owner.
7. The improvement is observed across repetitions, not only one successful run.
8. The effective Core and provider budgets are visible in the immutable task snapshot.

If no profile improves quality, keep the baseline and investigate selection, projection, prompt structure, or scenario coverage instead of increasing limits.

## Result and approval record

Complete this section only after running the benchmark. It is intentionally blank in the committed notebook.

- Baseline profile: `B6000-O6000`
- Selected profile: _pending benchmark_
- Repository revision: _pending benchmark_
- Model/provider snapshot: _pending benchmark_
- Quality evidence: _pending benchmark_
- Latency evidence: _pending benchmark_
- Cost evidence: _pending benchmark_
- Human decision to change production policy: _required_

No production budget is changed merely because this notebook exists or because a profile passes technical tests.